In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


# PASO 1. Buscar y cargar el CSV original.
nombre_archivo = "04LicitacionesEne.csv"
directorio_actual = Path.cwd()

ruta = None
for carpeta in [directorio_actual, *directorio_actual.parents]:
    candidata = carpeta / "data" / "raw" / nombre_archivo
    if candidata.is_file():
        ruta = candidata
        break

if ruta is None:
    raise FileNotFoundError(
        f"No se encontró {nombre_archivo} en data/raw/."
    )

df = pd.read_csv(ruta, sep=None, engine="python", encoding="latin-1")
df_limpio = df.copy()

print(f"Archivo leído: {ruta.relative_to(ruta.parents[2]).as_posix()}")
print(f"Filas originales: {len(df):,}")


# PASO 2. Quitar espacios al inicio y al final de los textos.
# Si un texto queda vacío, registrarlo como dato faltante.
columnas_texto = [
    "Proveedor", "Institucion", "UnidadCompra", "UnidadMedida", "NombreItem",
    "ResultadoOferta", "EstadoOferta", "EstadoLicitacion", "TipoLicitacion",
    "TamanoProveedor", "MonedaOferta",
]

ajustes_texto = {}
for columna in columnas_texto:
    original = df_limpio[columna].astype("string")
    limpio = original.str.strip().replace("", pd.NA)

    cambios = original.notna() & (limpio.isna() | original.ne(limpio))
    ajustes_texto[columna] = int(cambios.sum())
    df_limpio[columna] = limpio

print("\nAjustes de texto por columna")
display(pd.Series(ajustes_texto, name="Valores ajustados").to_frame())


# PASO 3. Convertir las fechas y los montos a sus tipos correctos.
columnas_fecha = ["FechaPublicacion", "FechaCierre", "FechaAdjudicacion"]
columnas_monto = ["MontoNetoOferta", "MontoTotalOferta"]

for columna in columnas_fecha:
    texto = df_limpio[columna].astype("string").str.strip().replace("", pd.NA)
    df_limpio[columna] = pd.to_datetime(
        texto, format="%d-%m-%Y %H:%M", errors="raise"
    )

for columna in columnas_monto:
    texto = df_limpio[columna].astype("string").str.strip().replace("", pd.NA)
    texto = texto.str.replace(",", ".", regex=False)
    df_limpio[columna] = pd.to_numeric(texto, errors="raise")


# PASO 4. Contar faltantes y revisar las fechas ausentes por estado.
# Conservamos la fecha faltante porque corresponde a un evento real.
print("\nFaltantes en fechas y montos")
display(
    df_limpio[columnas_fecha + columnas_monto]
    .isna()
    .sum()
    .to_frame("Faltantes")
)

df_limpio["SinFechaAdjudicacion"] = df_limpio["FechaAdjudicacion"].isna()

print("\nFechas de adjudicación faltantes por estado")
display(
    df_limpio.groupby("EstadoLicitacion", dropna=False)["SinFechaAdjudicacion"]
    .agg(Filas="size", SinFecha="sum")
)


# PASO 5. Marcar montos negativos y fechas fuera de orden para revisión.
df_limpio["MontoNegativo"] = df_limpio[columnas_monto].lt(0).any(axis=1)

df_limpio["FechasFueraDeOrden"] = (
    df_limpio["FechaCierre"].lt(df_limpio["FechaPublicacion"])
    | df_limpio["FechaAdjudicacion"].lt(df_limpio["FechaCierre"])
)

# Calcular el rango intercuartílico solo para ofertas en pesos chilenos.
# Los montos atípicos se conservan porque pueden ser válidos.
es_clp = df_limpio["MonedaOferta"].eq("CLP").fillna(False)
monto_total = df_limpio["MontoTotalOferta"]

q1, q3 = monto_total.loc[es_clp].quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

df_limpio["MontoAtipicoCLP"] = (
    es_clp & ((monto_total < limite_inferior) | (monto_total > limite_superior))
).fillna(False)

columnas_revision = [
    "SinFechaAdjudicacion", "MontoNegativo",
    "FechasFueraDeOrden", "MontoAtipicoCLP",
]

print("\nFilas marcadas para revisión")
display(df_limpio[columnas_revision].sum().to_frame("Filas marcadas"))


# PASO 6. Guardar la base limpia y volver a leerla para comprobarla.
assert df_limpio.index.equals(df.index), "Cambió el conjunto de filas."

carpeta_salida = ruta.parent.parent / "processed"
carpeta_salida.mkdir(parents=True, exist_ok=True)
salida = carpeta_salida / "04LicitacionesEne_limpio.csv"

df_limpio.to_csv(salida, sep=";", index=False, encoding="utf-8-sig")

verificacion = pd.read_csv(
    salida, sep=";", encoding="utf-8-sig", low_memory=False
)

assert verificacion.shape == df_limpio.shape, "Las dimensiones no coinciden."
assert verificacion.columns.equals(df_limpio.columns), "Las columnas no coinciden."

print(f"\nCSV guardado en: {salida.relative_to(ruta.parents[2]).as_posix()}")
print(f"Base verificada: {len(verificacion):,} filas y {verificacion.shape[1]} columnas.")

Archivo leído: data/raw/04LicitacionesEne.csv
Filas originales: 45,741

Ajustes de texto por columna


,Valores ajustados
Proveedor,420
Institucion,392
UnidadCompra,566
UnidadMedida,0
NombreItem,0
ResultadoOferta,0
EstadoOferta,0
EstadoLicitacion,0
TipoLicitacion,0
TamanoProveedor,0



Faltantes en fechas y montos


,Faltantes
FechaPublicacion,0
FechaCierre,0
FechaAdjudicacion,91
MontoNetoOferta,0
MontoTotalOferta,0



Fechas de adjudicación faltantes por estado


,Filas,SinFecha
EstadoLicitacion,,
Adjudicada,45570,0
Cerrada,95,91
Desierta (o art. 3 ó 9 Ley 19.886),72,0
Suspendida,4,0



Filas marcadas para revisión


,Filas marcadas
SinFechaAdjudicacion,91
MontoNegativo,1
FechasFueraDeOrden,0
MontoAtipicoCLP,7636



CSV guardado en: data/processed/04LicitacionesEne_limpio.csv
Base verificada: 45,741 filas y 78 columnas.
